In [ ]:
# Part 3 — Machine Learning: predict no-shows and choose the model honestly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

# Loading the dataset
df = pd.read_csv("/content/appointments_clean.csv")
X = df.drop(columns=["no_show", "appointment_id"], errors="ignore")
y = df["no_show"]

# Implementing train-test-split for the generated cleaned appointments dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Filtering the dataframe to keep columns containing numeric data types (such as integers and floats).
num_cols = X_train.select_dtypes(include=["number"]).columns
# Filtering the dataframe to keep the names of all non-numerical columns.
cat_cols = X_train.select_dtypes(exclude=["number"]).columns

# generating preprocessor object "ColumnTransformer" to apply data-cleaning steps to the numerical and categorical columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols), # "num" label generated, StandardScaler() rescales number to share similar range(mean=0 & std dev=1), num_cols-->numerical column names extracted that restrict restricts the scaler
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])  # cat shortname for categorical, OneHotEncoder creates a new binary column for every unique category as 0 or 1,  handle_unknown="ignore"--generate safety for new data(encoder fills all the category columns for that row with 0)
    # sparse_output=False(forces encoder to return a standard, readable NumPy array(table of 0 and 1), instead of hiding all 0 enteries to sparse matrix.
    # cat_cols--non-numerical column names that guide encoder to only transform specific columns and ignore numerical data.

# estimating imbalance ratio for binary classification in XGBClassifier
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(max_depth=4,learning_rate=0.05,scale_pos_weight=neg_count / pos_count,random_state=42
    )
}

# Introducing 5-fold stratified cross-validation (shuffled with a fixed seed) to preserve class balance across folds
# n_splits=5, splits training data into five equal-sized subsets (folds).
# n_splits=5, ensures that during cross_validate, the model trains 5 separate times: in each round, it trains on 4 folds (80%) and validates on the remaining 1 fold (20%),ensuring that every fold has served as the validation set once.
# shuffle=True, randomly mixes the indices within each class removing ordering bias.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Defing scoring metrices for evaluations, during cross validation step
scoring = ["accuracy", "precision", "recall", "roc_auc"]

# Compare models using ONLY the training set

for name, clf in models.items(): # loops different models to name (the string name of the model) and clf(classifier) variable assigned to the value
    pipe = Pipeline([("prep", preprocessor), ("model", clf)]) # initializes object "Pipeline" linking data preprocessing steps and clf(classifier) variable into into a single, unified workflow.
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring) # cross_validates splits training data into multiple subsets ("folds") that check model's ability to generalize unseen data.
    print(f"--- {name} ---") # print different model's name as stored in model names using for loop
    for metric in scoring:  # iterates through metric ("accuracy", "precision", "recall", "roc_auc")
        print(f"  {metric}: {scores[f'test_{metric}'].mean():.3f}")


In [ ]:
# Task 3.1 — Features, split and baseline
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load data (preserve exact row order)
df = pd.read_csv("/content/appointments_clean.csv")

# 2. Select only the allowed feature columns
features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]
X = df[features].copy()

# 3. Map reminder_sent (Yes/No -> 1/0) and one-hot encode categorical columns
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])

# 4. Define target variable
y = df["no_show"]

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Calculate and print metrics
train_size = len(X_train)
test_size = len(X_test)
test_no_shows = int(y_test.sum())

# Baseline: always predicting "Attended" (no_show == 0)
baseline_accuracy = (y_test == 0).mean()

print(f"Training set size: {train_size}")
print(f"Test set size: {test_size}")
print(f"Number of no-shows in test set: {test_no_shows}")
print(f"Baseline accuracy (always predicting Attended): {baseline_accuracy:.4f} ({baseline_accuracy:.2%})")


In [ ]:
# Task 3.2 — Logistic regression
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 1. Load data
df = pd.read_csv("/content/appointments_clean.csv")

# 1. Scale features (fit on training set only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 2. Fit Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

# 3. Generate predictions and probabilities on the test set
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# 4. Report test-set evaluation metrics
print("--- Test Set Evaluation Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 5. Print feature names alongside their coefficients in exact column order
print("\n--- Feature Coefficients (Scaled) ---")
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})
print(coef_df.to_string(index=False))

# 6. Compute scaled vs. unscaled odds ratios for reminder_sent
rem_idx = list(X.columns).index("reminder_sent")
rem_coef_scaled = model.coef_[0][rem_idx]
rem_std = scaler.scale_[rem_idx]

odds_ratio_scaled = np.exp(rem_coef_scaled)
odds_ratio_unscaled = np.exp(rem_coef_scaled / rem_std)

print("\n--- Odds Ratio for reminder_sent ---")
print(f"Scaled np.exp(coef):              {odds_ratio_scaled:.4f}")
print(f"Unscaled np.exp(coef / scale_):   {odds_ratio_unscaled:.4f}")


In [ ]:
# Task 3.3 — Decision tree and overfitting

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv("/content/appointments_clean.csv")

# 1. Decision Tree with no depth limit
dt_unlimited = DecisionTreeClassifier(random_state=42)
dt_unlimited.fit(X_train, y_train)

train_acc_unlim = accuracy_score(y_train, dt_unlimited.predict(X_train))
test_acc_unlim = accuracy_score(y_test, dt_unlimited.predict(X_test))

# 2. Decision Tree with max_depth=3
dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

train_acc_d3 = accuracy_score(y_train, dt_depth3.predict(X_train))
test_acc_d3 = accuracy_score(y_test, dt_depth3.predict(X_test))

# 3. Report results
print("--- Decision Tree (No Depth Limit) ---")
print(f"Training Accuracy: {train_acc_unlim:.4f}")
print(f"Test Accuracy:     {test_acc_unlim:.4f}")
print(f"Gap (Train - Test): {train_acc_unlim - test_acc_unlim:.4f}\n")

print("--- Decision Tree (max_depth=3) ---")
print(f"Training Accuracy: {train_acc_d3:.4f}")
print(f"Test Accuracy:     {test_acc_d3:.4f}")
print(f"Gap (Train - Test): {train_acc_d3 - test_acc_d3:.4f}")


In [ ]:
# Task 3.4 — Random forest and a comparison table
# Load data
df = pd.read_csv("/content/appointments_clean.csv")

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# 1. Fit Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 2. Generate test predictions and probabilities
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# 3. Compute Random Forest metrics
rf_acc = accuracy_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("--- Random Forest Test Set Metrics ---")
print(f"Accuracy: {rf_acc:.4f}")
print(f"Recall:   {rf_rec:.4f}")
print(f"ROC-AUC:  {rf_auc:.4f}")

# 4. Extract top three feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n--- Top 3 Feature Importances ---")
print(importances.head(3).to_string())

# 5. Build Comparison Table across Logistic Regression, Depth-3 Tree, and Random Forest
# (Assumes `model` and `X_test_s` from Logistic Regression, and `dt_depth3` from Decision Tree are in memory)
lr_pred = model.predict(X_test_s)
lr_prob = model.predict_proba(X_test_s)[:, 1]

dt3_pred = dt_depth3.predict(X_test)
dt3_prob = dt_depth3.predict_proba(X_test)[:, 1]

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree (max_depth=3)", "Random Forest (n_estimators=100)"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt3_pred),
        rf_acc
    ],
    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, dt3_pred),
        rf_rec
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, lr_prob),
        roc_auc_score(y_test, dt3_prob),
        rf_auc
    ]
})

print("\n--- Model Comparison Summary ---")
print(comparison_df.to_string(index=False, float_format="%.4f"))


In [ ]:
# Task 3.5 — Tune, choose the model and the threshold, then test once
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Load and prepare data to ensure X_train, y_train, X_test, y_test are defined in this cell
df = pd.read_csv("/content/appointments_clean.csv")
features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]
X = df[features].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]

# (a) GridSearchCV on the training set

param_grid = {"n_estimators": [50, 100, 200], "max_depth": [3, 5, 7, None]}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)
grid.fit(X_train, y_train)

print("--- (a) GridSearchCV Results ---")
print(f"Best Params: {grid.best_params_}")
print(f"Best CV ROC-AUC: {grid.best_score_:.4f}\n")

# (b) Compare 5-fold CV ROC-AUC across all 4 models on X_train (no test set)

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])
dt3 = DecisionTreeClassifier(max_depth=3, random_state=42)
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_tuned = grid.best_estimator_

models = {
    "Logistic Regression": lr_pipe,
    "Decision Tree (max_depth=3)": dt3,
    "Random Forest (default)": rf_base,
    "Random Forest (tuned)": rf_tuned,
}

cv_scores = {}
print("--- (b) 5-Fold Cross-Validated ROC-AUC Comparison ---")
for name, clf in models.items():
    if name == "Random Forest (tuned)":
        score = grid.best_score_  # Already computed by 5-fold GridSearchCV
    else:
        score = cross_val_score(clf, X_train, y_train, cv=5, scoring="roc_auc").mean()
    cv_scores[name] = score
    print(f"{name:30s}: {score:.4f}")

winner_name = max(cv_scores, key=cv_scores.get)
winner_model = models[winner_name]
print(f"\nWinner Selected: {winner_name} (CV ROC-AUC = {cv_scores[winner_name]:.4f})\n")


# (c) Out-of-fold probabilities & threshold comparison (0.5 vs. 0.35)
oof_probs = cross_val_predict(
    winner_model, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]

print("--- (c) Threshold Comparison on Training OOF Predictions ---")
threshold_results = {}
for thresh in [0.5, 0.35]:
    oof_preds = (oof_probs >= thresh).astype(int)
    prec = precision_score(y_train, oof_preds)
    rec = recall_score(y_train, oof_preds)
    f1 = f1_score(y_train, oof_preds)
    flagged = int(oof_preds.sum())
    threshold_results[thresh] = f1
    print(f"Threshold {thresh:.2f} -> Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f} | Flagged: {flagged}")

chosen_threshold = max(threshold_results, key=threshold_results.get)
print(f"\nChosen Threshold (higher F1): {chosen_threshold}\n")

# (d) Fit winner on full X_train and evaluate ONCE on X_test

winner_model.fit(X_train, y_train)
test_probs = winner_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= chosen_threshold).astype(int)

test_prec = precision_score(y_test, test_preds)
test_rec = recall_score(y_test, test_preds)
test_flagged = int(test_preds.sum())
test_auc = roc_auc_score(y_test, test_probs)

print("--- (d) Final One-Time Evaluation on Test Set ---")
print(f"Chosen Threshold: {chosen_threshold}")
print(f"Test Precision:   {test_prec:.4f}")
print(f"Test Recall:      {test_rec:.4f}")
print(f"Number Flagged:   {test_flagged} (out of {len(y_test)} test rows)")
print(f"Test ROC-AUC:     {test_auc:.4f}\n")

# (e) Save test_risk_scores.csv
output_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"],
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_preds
})

output_df.to_csv("test_risk_scores.csv", index=False)
print("--- (e) Saved test_risk_scores.csv ---")
print(output_df.head())
